# Import libraries

In [183]:
import pandas as pd

# Get the data

In [184]:
# Load the dataset from the CSV file
dataset = pd.read_csv('../data/book700k-800k.csv')
df = pd.DataFrame({'Id': dataset['Id'],
                        'Name': dataset['Name'],
                        'Authors': dataset['Authors'],
                        'Publish year': dataset['PublishYear'],
                        'Rating': dataset['Rating'],
                        'Description': dataset['Description']})
df['Text feature'] = (df['Authors'].fillna('') + ' ' +
                      df['Name'].fillna('') + ' ' +
                      df['Description'].fillna('') +
                      df['Description'].fillna('') +
                      df['Description'].fillna(''))

# Display the first few rows of the dataset
df.head


<bound method NDFrame.head of            Id                                               Name  \
0      700000  A Passion to Preserve: Gay Men as Keepers of C...   
1      700002  Culture Keepers-Florida: Oral History of the A...   
2      700003  Holiday Favorites: The Best of the Williams-So...   
3      700004  Soups, Salads & Starters: the Best of Williams...   
4      700005                              Breakfasts & Brunches   
...       ...                                                ...   
54268  799991           Piano Concerto Highlights for Solo Piano   
54269  799993  Noggin King of the Nogs (The Sagas of Noggin t...   
54270  799994  No Greater Glory: The Four Immortal Chaplains ...   
54271  799996  The White Company by Arthur Conan Doyle, Ficti...   
54272  799997                    Livewire Real Lives Dawn Fraser   

                     Authors  Publish year  Rating  \
0               Will Fellows          2005    3.75   
1      Deborah Johnson-Simon          2006   

# Data preprocessing

In [185]:
import re
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

# Initialize the stopwords
stop_words = set(stopwords.words('english'))

def preprocessing_text(text):
    # Convert the input text to string
    text = str(text)
    
    # Convert text to lowercase
    text = text.lower()
    
    # Remove special characters and digits and replace them with a spcae
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    
    # Tokenize the text
    tokens = nltk.word_tokenize(text)
    
    # Remove stop words
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatize the tokens (convert words into their base dictionary form, ex. cats->cat)
    tokens = [lemmatizer.lemmatize(word, pos='v') for word in tokens]
    
    # Return the processed text as a string
    return " ".join(tokens)


def preprocess_dataframe(df, column_name):
    df[column_name]= df[column_name].apply(preprocessing_text)
    return df


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\thlam\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\thlam\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\thlam\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Get text features using tf-idf
`from sklearn.feature_extraction.text import TfidfVectorizer`

In [186]:
from sklearn.feature_extraction.text import TfidfVectorizer

def get_text_feature(df):
    text_features = df['Text feature']
    # Text feature
    tfidf = TfidfVectorizer(stop_words="english",
                            strip_accents='ascii',
                            token_pattern=r'\w+')

    tfidf_matrix = tfidf.fit_transform(text_features)
    tfidf.get_feature_names_out()
    
    return tfidf_matrix

# Calculate cosine similarity
`from sklearn.metrics.pairwise import cosine_similarity`

In [187]:
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity_of_a_target_book(book, tfidf_matrix):
    """Calculate the cosine similarity between a target book and all books
    in the TF-IDF matrix.

    Args:
        book (scipy.sparse.csr_matrix): 
            A single TF-IDF row vector representing the target book.
            Shape should be (1, n_features).
        tfidf_matrix (scipy.sparse.csr_matrix_): 
            TF-IDF matrix containing all book vectors.
            Shape should be (n_books, n_features).

    Returns:
        numpy.ndarray: A 2D array containing cosine similarity scores between the target book and 
        every book in the TF-IDF matrix.
        Shape will be (1, n_books).
    """
    X = book
    Y = tfidf_matrix
    cs = cosine_similarity(X, Y)
    return cs

# Add indices to the similarity array and sort it

In [188]:
def add_indices(sim_array, tfidf_matrix):
    indexed_array = []
    for i, s in zip(range(tfidf_matrix.shape[0]), sim_array[0]):
        indexed_array.append((i, s))
        
    sorted_array = sorted(indexed_array, key=lambda x: x[1], reverse=True)
    return sorted_array[1:11]
    


# Get the book names from the indices

In [189]:
def get_book_name(rec_books):
    rec_books_info = []
    for i in rec_books:
        df['Similarity'] = i[1]
        rec_books_info.append(df.iloc[i[0], :])
    return pd.DataFrame(rec_books_info)

# Put everything together

In [190]:
def get_token_matrix(df):
    preprocessed_text = preprocess_dataframe(df=df, column_name='Text feature')
    matrix = get_text_feature(preprocessed_text)
    return matrix

matrix = get_token_matrix(df=df)

In [202]:
def get_rec_books(book_idx, df, matrix):
    selected_book = df.iloc[book_idx:book_idx+1, :]
    book_sim = calculate_cosine_similarity_of_a_target_book(book=matrix[book_idx], tfidf_matrix=matrix)
    book_sim = add_indices(book_sim, tfidf_matrix=matrix)
    result = get_book_name(rec_books=book_sim)
    
    return selected_book, result

selected_book, rec_books = get_rec_books(df=df, matrix=matrix, book_idx=421)

print(f"Selected book: {selected_book['Name'].iloc[0]}")
print(f"Selected book description: '{selected_book['Description'].iloc[0]}'\n")
print("10 similar books:")
for name, s in zip(rec_books['Name'], rec_books['Similarity']): 
    print(f"{name}\nSimilarity: {s*100:.2f}%\n")
    

Selected book: Hack Proofing Your Network
Selected book description: 'A new edition the most popular Hack Proofing book around!<br /><br />IT professionals who want to run secure networks, or build secure software, need to know about the methods of hackers. The second edition of the best seller <i>Hack Proofing Your Network</i>, teaches about those topics, including: - The Politics, Laws of Security, Classes of Attack, Methodology, Diffing, Decrypting, Brute Force, Unexpected Input, Buffer Overrun, Sniffing, Session Hijacking, Spoofing, Server Holes, Client Holes, Trojans and Viruses, Reporting Security Problems, Choosing Secure Systems The central idea of this book is that it's better for you to find the holes in your network than it is for someone else to find them, someone that would use them against you. The complete, authoritative guide to protecting your Windows 2000 Network.<br /><br /><br /><br />Updated coverage of an international bestseller and series flagship<br />Covers mo

In [195]:
rec_books

,Id,Name,Authors,Publish year,Rating,Description,Text feature,Similarity
19381,735727,"Nobel Prize, The: The First 100 Years",Agneta Wallin Levinovitz,2001,2.67,"The Nobel Prize, as founded in Alfred Nobel's ...",agneta wallin levinovitz nobel prize first yea...,0.388685
12735,723495,Beyond the Cold War: New Dimensions in Interna...,Geir Lundestad,1993,0.00,In December 1991 the Nobel prizes celebrated t...,geir lundestad beyond cold war new dimension i...,0.357543
2079,703876,Step 2 What a Day!,J. Alexander,2003,0.00,NaN,j alexander step day,0.339714
1446,702759,Women Nobel Peace Prize Winners,Anita Price Davis,2006,3.83,To benefit humanity - an ironic epitaph for th...,anita price davis women nobel peace prize winn...,0.324359
19392,735745,"The Road to Stockholm: Nobel Prizes, Science, ...",István Hargittai,2002,3.18,The Nobel Prize is by far the highest recognit...,istv n hargittai road stockholm nobel prize sc...,0.317855
35823,765925,Alexander Technique: Original Writings of F.M....,Danny McGowan,1997,4.25,Details Alexander's principles for achieving c...,danny mcgowan alexander technique original wri...,0.300241
23661,743696,Alexander: Destiny and Myth,Claude Mossé,2004,3.47,Few figures from history have aroused as much ...,claude moss alexander destiny myth figure hist...,0.284389
1448,702761,"Pioneers of Science, Nobel Prizewinners in Phy...",Robert L. Weber,1988,5.00,This second edition of ^IPioneers of Science f...,robert l weber pioneer science nobel prizewinn...,0.276271
19384,735730,Alfred Nobel and the Story of the Nobel Prize ...,John Bankston,2003,3.00,NaN,john bankston alfred nobel story nobel prize g...,0.273461
35500,765391,Wonders of the Lost Age,Alan Alexander,2006,3.46,NaN,alan alexander wonder lose age,0.272884


In [193]:
selected_book = pd.DataFrame(selected_book)
selected_book

,Id,Name,Authors,Publish year,Rating,Description,Text feature
46201,785222,"Barons, Rebels & Romantics: The Fitzgeralds Fi...",Alan John Fitzgerald,2004,0.0,NaN,alan john fitzgerald barons rebel romantics fi...
